# RAG pipeline example

Educational disease-symptom RAG — **retrieval + rerank** (BM25, k-NN, hybrid + RRF, then cross-encoder rerank).

## Prerequisites

1. `.env` with `OPENSEARCH_*` credentials (project root)
2. Index bootstrapped:
   ```bash
   uv run python -m src.migrations.init_db upgrade
   uv run python -m src.migrations.migrate_ddxplus_index upgrade
   ```
3. Documents indexed in the `diseases` alias (**ingest team** — retrieval assumes data exists)
4. Run from project root with the project venv:

```bash
cd disease-diagnosis-rag-system
uv sync
```
then run the notebook. First BGE/reranker calls download models into `models/` (lazy, singleton).

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "services" / "rag").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists(), (
    "Run this notebook from the repo root or notebooks/ folder"
)


os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: /home/mitu/projects/disease-diagnosis-rag-system


In [ ]:
from src.db.vector_db.opensearch import get_opensearch_client
from src.services.inference.embeddings.service import TextEmbeddingService
from src.services.inference.reranker.service import RerankerService
from src.services.rag import RAGService, Retriever
from src.services.rag.preprocess import preprocess_query
from src.services.rag.schemas import (
    Bm25RetrieveRequest,
    HybridRetrieveRequest,
    RetrievalMode,
    RetrieveExperimentRequest,
    RetrieveResult,
    VectorRetrieveRequest,
)
from src.settings import settings

## Configuration

Non-secret defaults loaded from `src/settings.py` and `.env`.

In [ ]:
print("Index alias:", settings.RETRIEVE_INDEX_ALIAS)
print("Search pipeline:", settings.CURRENT_SEARCH_PIPELINE)
print("Retrieve top_k:", settings.RETRIEVE_TOP_K)
print("Rerank top_k:", settings.RERANK_TOP_K)
print("Embedding model path:", settings.embedding_model_path)
print("Reranker model path:", settings.reranker_model_path)
print("OpenSearch host:", settings.OPENSEARCH_HOST)

## Step 1 — Query preprocessing

Normalize symptom tokens and expand synonyms before search (`tired` → `fatigue`).

In [ ]:
raw_query = "I am tired, have fever and cough"
search_query = preprocess_query(raw_query)

print("Raw:", raw_query)
print("Preprocessed:", search_query)

## Step 2 — Initialize services

First model calls download/load BGE-small-en-v1.5 and bge-reranker-base (lazy, singleton).

- **`Retriever`** — low-level search APIs and composable `rerank()` (experiments stay retrieval-only unless you inject a reranker)
- **`RAGService`** — production path: `query()` → hybrid retrieve (top 20) → rerank (top 5)

In [ ]:
client = get_opensearch_client()
embed_service = TextEmbeddingService()
rerank_service = RerankerService()

# Retrieval-only retriever for BM25 / k-NN / hybrid experiments
retriever = Retriever(client=client, embed_service=embed_service)

# Retriever with reranker for composable retrieve → rerank
retriever_with_rerank = Retriever(
    client=client,
    embed_service=embed_service,
    rerank_service=rerank_service,
)

QUERY = "fever cough fatigue"
TOP_K = 5  # display limit for experiment cells

In [ ]:
def show_hits(result: RetrieveResult, limit: int = 10) -> None:
    print(f"took_ms={result.took_ms}  hits={len(result.hits)}")
    for hit in result.hits[:limit]:
        score = f"{hit.score:.4f}" if hit.score is not None else "n/a"
        print(
            f"{hit.rank:>2}. {hit.disease:<24} "
            f"score={score}  symptoms={hit.symptoms}"
        )

## Step 3a — BM25 (lexical)

`match` on `keyword_text` with `operator: or`.

In [ ]:
bm25_result = retriever.search_bm25(
    Bm25RetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== BM25 ===")
show_hits(bm25_result)

## Step 3b — k-NN (semantic)

BGE query embedding (384-dim, cosine) on `embedding` field.

In [ ]:
vector_result = retriever.search_vector(
    VectorRetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== k-NN ===")
show_hits(vector_result)

## Step 3c — Hybrid + RRF (MVP default)

Single OpenSearch `hybrid` query fused server-side via `hybrid-rrf` pipeline.

In [ ]:
hybrid_result = retriever.search_hybrid(
    HybridRetrieveRequest(query=QUERY, top_k=TOP_K)
)
print("=== Hybrid + RRF ===")
show_hits(hybrid_result)

## Step 4 — Compare all modes (experiment runner)

Same query through BM25, k-NN, and hybrid side by side.

In [ ]:
comparison = retriever.run_experiment(
    RetrieveExperimentRequest(query=QUERY, top_k=TOP_K)
)

for mode, mode_result in comparison.results.items():
    print(f"\n=== {mode.value.upper()} (total_hits={mode_result.total_hits}) ===")
    show_hits(mode_result, limit=3)

## Step 5 — Production path (retrieve + rerank)

Hybrid search returns up to `RETRIEVE_TOP_K` (20) hits. Cross-encoder reranking keeps `RERANK_TOP_K` (5).

Each hit exposes `passage_text` — symptom-first text passed to the reranker (`Disease: …`, optional `Symptoms: …`, then description). Retrieval and rerank both preprocess the query when `preprocess=True` (default).

In [ ]:
# Composable: retrieve then rerank
retrieved = retriever_with_rerank.retrieve(QUERY)
print(f"Before rerank: {len(retrieved.hits)} hits")
if retrieved.hits:
    print("Sample passage_text:", retrieved.hits[0].passage_text[:120], "...")

reranked = retriever_with_rerank.rerank(
    QUERY, retrieved, top_k=settings.RERANK_TOP_K
)
print("\n=== After rerank (cross-encoder scores) ===")
show_hits(reranked)

# Facade — same flow as above
rag = RAGService(
    embed_service=embed_service,
    rerank_service=rerank_service,
)
result = rag.query(QUERY)
print("\n=== RAGService.query() ===")
show_hits(result)

## Optional — Inspect OpenSearch request body

Enable debug bodies on the experiment request to see the Query DSL sent to OpenSearch.

In [ ]:
import json

debug = retriever.run_experiment(
    RetrieveExperimentRequest(
        query=QUERY,
        top_k=TOP_K,
        include_opensearch_body=True,
    )
)

hybrid_debug = debug.results[RetrievalMode.HYBRID]
print(json.dumps(hybrid_debug.opensearch_body, indent=2)[:2000])

## Next steps

- **Ingest** — index disease records into `diseases` (see `src/services/rag/ingest.py` stub)
- **Generate** — Qwen3 8B with reranked context
- **API** — FastAPI endpoint wrapping `RAGService.query()`